# Iterative Workflow in LangGraph

A tweet writer that critiques and rewrites its own output — the **evaluator-optimizer** loop.

- **generate** &rarr; write a first draft tweet on the topic
- **evaluate** &rarr; an LLM judge returns structured `approved` / `needs_improvements` + feedback
- **optimize** &rarr; rewrite the tweet using that feedback, then loop back to **evaluate**

A **conditional edge** closes the cycle: the graph exits when the judge approves or when
`max_iterations` is hit, so the loop always terminates.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated , Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
import operator

In [6]:
generative_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")

In [ ]:
class TweetEvaluationSchema(BaseModel):
    evaluation: Literal["approved", "needs_improvements"]
    feedback: str  

In [12]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)

In [7]:
#state
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvements"]
    feedback: str
    iteration: int
    max_iterations: int

In [21]:
def generate_tweet(state: TweetState):
    #prompt
    message = [
        SystemMessage(content=f"You are a helpful assistant. Generate a tweet about {state['topic']} in 50 words or less."),
        HumanMessage(content=f"Please generate a tweet about {state['topic']}.")
    ]

    # call llm
    response = generative_llm(message)


    # return response
    return {"tweet": response}

In [22]:
def evaluate_tweet(state: TweetState):
    #prompt
    message = [
        SystemMessage(content=f"You are a helpful assistant. Evaluate the following tweet: {state['tweet']}. Provide feedback and indicate if it is approved or needs improvements."),
        HumanMessage(content=f"Please evaluate the tweet: {state['tweet']}.")
    ]

    # call llm
    response = structured_evaluator_llm.invoke(message).content

    # parse response
    evaluation = response.evaluation
    feedback = response.feedback

    # return response
    return {"evaluation": evaluation, "feedback": feedback}

In [23]:
def optimize_tweet(state: TweetState):
    #prompt
    message = [
        SystemMessage(content=f"You are a helpful assistant. Optimize the following tweet based on the feedback: {state['feedback']}."),
        HumanMessage(content=f"Please optimize the tweet: {state['tweet']} based on the feedback: {state['feedback']}.")
    ]

    # call llm
    response = optimizer_llm.invoke(message).content
    iteration = state['iteration'] + 1

    # return response
    return {"tweet": response, "iteration": iteration}

In [24]:
def route_evaluation(state: TweetState):
    if state['evaluation'] == "approved" or state['iteration'] >= state['max_iterations']:
        return 'approved'
    else:
        return 'needs_improvements'

In [ ]:
graph = StateGraph(TweetState)

#nodes 

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)


#edges
graph.add_edge(START , 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvements': 'optimize'})
graph.add_edge('optimize', 'evaluate')
workflow = graph.compile()


In [ ]:
initial_state = {
    "topic": "Artificial Intelligence",
    "tweet": "",
    "evaluation": "",
    "feedback": "",
    "iteration": 0,
    "max_iterations": 3
}   

final_state = workflow.invoke(initial_state)
print(final_state)